# Модуль 19.1 — журнал проверки: будильники для worker и watcher

Этот ноутбук ничего не устанавливает и не обращается к модели. Он — журнал вашей лабораторки: кроны, вотчдог и gateway вы уже завели по шагам ДЗ (лекция, раздел «Домашнее задание»), а ноутбук проверяет результат subprocess-вызовами `hermes` и чтением файлов `~/.hermes` — и печатает вердикты OK/FAIL.

Что проверяем:

- пререквизиты: Hermes установлен, профили worker и watcher из ДЗ модуля 19 существуют;
- крон worker: `every` + `Repeat: ∞` (не `once in`);
- вотчдог watcher: no-agent, скрипт — реальный файл в `scripts/` профиля, не симлинк;
- тикер: gateway обоих профилей на месте, heartbeat свежий, отчёты складываются в `cron/output/`.

Бонусный шаг ДЗ — событийная побудка через `webhook subscribe` с гейт-скриптом — ноутбуком не проверяется.

**Только локальный запуск.** Colab и Kaggle не увидят ваш `~/.hermes` — работа целиком на вашей машине. Все проверки keyless: ни одного LLM-вызова, ни одного ключа.


In [ ]:
# Ячейка 1. Пререквизиты: Hermes и оба профиля из ДЗ модуля 19
import json, shutil, subprocess, time
from datetime import datetime
from pathlib import Path

HERMES = shutil.which("hermes")
HERMES_DIR = Path.home() / ".hermes"
PROFILES = HERMES_DIR / "profiles"

# Если вы назвали профили иначе - поменяйте здесь:
WORKER = "worker"
WATCHER = "watcher"

RESULTS = {}  # копилка вердиктов для итоговой сводки

def verdict(name, ok, detail=""):
    RESULTS[name] = bool(ok)
    print(("OK   - " if ok else "FAIL - ") + name + (f" ({detail})" if detail else ""))
    return bool(ok)

def hermes(*args, timeout=120):
    """Вызов hermes через subprocess; возвращает (код возврата, вывод)."""
    if not HERMES:
        return None, ""
    r = subprocess.run([HERMES, *args], capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout or "") + (r.stderr or "")

if not HERMES:
    print("FAIL - команда hermes не найдена в PATH.")
    print("Это ДЗ - продолжение: Hermes ставится в ДЗ модуля 19 (папка notebooks/module-19-hermes")
    print("в этом же репо). Если Hermes установлен, но не виден - бинарь лежит в ~/.local/bin:")
    print('добавьте каталог в PATH (export PATH="$HOME/.local/bin:$PATH"), перезапустите терминал')
    print("и запустите jupyter из него заново.")
    raise SystemExit("сначала ДЗ модуля 19: hermes не в PATH")

rc, out = hermes("profile", "list")
print(out.strip())
print()
have = WORKER in out and WATCHER in out
verdict("профили worker и watcher существуют (ДЗ модуля 19)", have,
        f"ищем имена: {WORKER}, {WATCHER}")
if not have:
    print()
    print("Профилей нет - это ДЗ опирается на ДЗ модуля 19: вернитесь в notebooks/module-19-hermes,")
    print("заведите worker и watcher и возвращайтесь. Если профили названы иначе -")
    print("поправьте WORKER/WATCHER выше и перезапустите ноутбук.")
    raise SystemExit("сначала ДЗ модуля 19: нет профилей worker/watcher")


In [ ]:
# Ячейка 2. Крон worker: every + Repeat: бесконечность, никаких once in
rc, out = hermes("-p", WORKER, "cron", "list")
print(out.strip())
print()

jobs_path = PROFILES / WORKER / "cron" / "jobs.json"
ok_file = verdict("jobs.json существует", jobs_path.is_file(), str(jobs_path))

jobs = []
if ok_file:
    data = json.loads(jobs_path.read_text(encoding="utf-8"))
    jobs = data.get("jobs", data if isinstance(data, list) else [])

verdict("в jobs.json есть хотя бы один джоб", len(jobs) >= 1,
        "пустой jobs.json при живом тикере - тот самый замаскированный отказ из лекции")

sched_ok = "every" in out and "once in" not in out
verdict("расписание - every, не once in", sched_ok,
        "строка Schedule: once in - неисправность, а не настройка")

rep_ok = bool(jobs) and all((j.get("repeat") or {}).get("times") is None for j in jobs)
verdict("Repeat без лимита (repeat.times == null)", rep_ok,
        "без флага --repeat N у interval-джоба times = null, в cron list это Repeat: беск.")

if not sched_ok:
    print()
    print('Подсказка: hermes -p worker cron create "every 10m" "<промпт>" --name loop ;')
    print("once in-джоб исчезает после срабатывания - цепочка умирает на первом холостом ходе.")


In [ ]:
# Ячейка 3. Вотчдог watcher: no-agent + реальный файл в scripts/
rc, out = hermes("-p", WATCHER, "cron", "list")
print(out.strip())
print()

jobs_path = PROFILES / WATCHER / "cron" / "jobs.json"
jobs = []
if jobs_path.is_file():
    data = json.loads(jobs_path.read_text(encoding="utf-8"))
    jobs = data.get("jobs", data if isinstance(data, list) else [])

verdict("у watcher есть хотя бы один джоб", len(jobs) >= 1)

raw = json.dumps(jobs, ensure_ascii=False)
no_agent_ok = '"no_agent": true' in raw or "no-agent" in out.lower() or "no agent" in out.lower()
verdict("джоб no-agent: LLM не вызывается вовсе", no_agent_ok,
        "скрипт и есть джоб - classic watchdog pattern")

scripts_dir = PROFILES / WATCHER / "scripts"
files = sorted(scripts_dir.iterdir()) if scripts_dir.is_dir() else []
verdict("в scripts/ профиля есть скрипт", len(files) >= 1, str(scripts_dir))

links = [p.name for p in files if p.is_symlink()]
verdict("ни один скрипт не симлинк", bool(files) and not links,
        "симлинки: " + ", ".join(links) if links else "test -L прошёл бы")

if links:
    print()
    print("Подсказка: симлинк тикер отклонит с 'Blocked: script path resolves outside the")
    print("scripts directory', хотя ручной прогон проходит. Скопируйте файл по-настоящему.")


In [ ]:
# Ячейка 4. Здоровье тикера: gateway, heartbeat, отчёты - диагностическая лесенка
rc, out = hermes("gateway", "list")
print(out.strip())
print()
verdict("worker и watcher видны в gateway list", WORKER in out and WATCHER in out,
        "тикер крона живёт в gateway-процессе профиля")

rc2, out2 = hermes("-p", WORKER, "cron", "status")
print(out2.strip())
print()

now = time.time()
hb_fail = False
for name in (WORKER, WATCHER):
    hb = PROFILES / name / "cron" / "ticker_heartbeat"
    if hb.is_file():
        age = int(now - hb.stat().st_mtime)
        verdict(f"тикер {name} жив (heartbeat моложе 5 минут)", age < 300, f"{age} с назад")
        hb_fail = hb_fail or age >= 300
    else:
        verdict(f"тикер {name} жив", False, "нет файла ticker_heartbeat")
        hb_fail = True

if hb_fail:
    print()
    print("Подсказка: тикер живёт в gateway-процессе профиля. hermes -p worker gateway install,")
    print("затем hermes -p worker gateway start (то же для watcher); на WSL, в Docker и Termux -")
    print("hermes -p worker gateway run в отдельном терминале. Проверка - hermes gateway list;")
    print("при странностях - hermes doctor (есть и --fix).")

out_dir = PROFILES / WORKER / "cron" / "output"
reports = sorted(out_dir.rglob("*.md"), key=lambda p: p.stat().st_mtime) if out_dir.is_dir() else []
if reports:
    last_age = int(now - reports[-1].stat().st_mtime)
    print(f"Последний отчёт worker: {reports[-1].name}, {last_age} с назад.")
    print("Лесенка из лекции: живой heartbeat при старых отчётах = 'задания нет', а не 'всё работает'.")
else:
    print("Отчётов в cron/output/ worker пока нет - если джоб создан только что, дождитесь")
    print("первого тика (тикер проверяет расписание раз в 60 секунд).")


In [ ]:
# Ячейка 5. Итоговая сводка
print("=== Итоговая сводка - приложите этот вывод в чат курса как [Модуль 19.1, ДЗ] ===")
print("Дата прогона:", datetime.now().isoformat(timespec="seconds"))
fails = [k for k, ok in RESULTS.items() if not ok]
for k, ok in RESULTS.items():
    print(("OK   " if ok else "FAIL ") + "- " + k)
total = len(RESULTS)
print()
print(f"Итого: {total - len(fails)}/{total} проверок зелёные"
      + ("" if not fails else " - разберитесь с FAIL выше и перезапустите ноутбук"))


## Итог

Чек-лист перед сдачей — он же критерии приёма из лекции:

- [ ] Крон worker — `every` с `Repeat: ∞`; «once in» в `jobs.json` нет.
- [ ] Вотчдог watcher — no-agent, скрипт лежит реальным файлом в `scripts/` профиля, не симлинком.
- [ ] `hermes gateway list` показывает worker и watcher; `ticker_heartbeat` обоих профилей свежий. Отчёты в `cron/output/` ноутбук только печатает — их появление после первых тиков проверяете глазами.
- [ ] `Run all` прошёл целиком, итоговая сводка без FAIL.

Если что-то FAIL — диагностическая лесенка из лекции: `hermes cron list` у профиля → свежесть `cron/ticker_heartbeat` против последнего файла в `cron/output/` → последний `## Response` в отчёте → `hermes gateway list` (тикер живёт в gateway) → `hermes doctor`. Команды и дефолты — по [документации Hermes Agent](https://hermes-agent.nousresearch.com/docs/).

**Что приложить в чат курса:** вывод итоговой сводки из Ячейки 5, пометка `[Модуль 19.1, ДЗ]`. Преподаватель не нужен: все пункты закрыты — домашка сдана.
